In [4]:
pip install gensim

   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
    --------------------------------------- 0.5/24.4 MB 1.3 MB/s eta 0:00:19
   - -------------------------------------- 0.8/24.4 MB 1.3 MB/s eta 0:00:19
   - -------------------------------------- 1.0/24.4 MB 1.4 MB/s eta 0:00:17
   -- ------------------------------------- 1.6/24.4 MB 1.6 MB/s eta 0:00:15
   --- ------------------------------------ 1.8/24.4 MB 1.6 MB/s eta 0:00:15
   --- ------------------------------------ 2.1/24.4 MB 1.5 MB/s eta 0:00:16
   --- ------------------------------------ 2.1/24.4 MB 1.5 MB/s eta 0:00:16
   --- ------------------------------------ 2.4/24.4 MB 1.4 MB/s eta 0:00:16
   --- ------------------------------------ 2.4/24.4 MB 1.4 MB/s eta 0:00:16
   ---- ----------------------------------- 2.6/24.4 MB 1.2 MB/s eta 0:00:18
   ---- ----------------------------------- 2.6/24.4 MB 1.2 MB/s eta 0:00:18
   ---- -----

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report
import numpy as np

In [6]:
import pandas as pd

df = pd.read_csv(r'D:\vs code projects\yt_comment_analyzer\data\processed\final_data.csv')
df.shape

(199508, 2)

In [7]:
# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

In [9]:
X = df['clean_comment']
y = df['category']

In [10]:
# Step 3: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [11]:
# Tokenize the sentences for Word2Vec training
X_train_tokenized = [sentence.split() for sentence in X_train]
X_test_tokenized = [sentence.split() for sentence in X_test]

In [12]:
# Train Word2Vec model
word2vec_model = Word2Vec(sentences=X_train_tokenized, vector_size=300, window=5, min_count=1, workers=4, sg=1)  # sg=1 for Skip-Gram model

In [13]:
# Generate the vector representation for each comment
def vectorize_comments(tokenized_comments, word2vec_model):
    vectorized_comments = []
    for tokens in tokenized_comments:
        vectors = [word2vec_model.wv[token] for token in tokens if token in word2vec_model.wv]
        if len(vectors) > 0:
            vectorized_comments.append(np.mean(vectors, axis=0))  # Mean of all word vectors in the comment
        else:
            vectorized_comments.append(np.zeros(word2vec_model.vector_size))  # If no words match, use a zero vector
    return np.array(vectorized_comments)

In [14]:
# Vectorize train and test comments
X_train_word2vec = vectorize_comments(X_train_tokenized, word2vec_model)
X_test_word2vec = vectorize_comments(X_test_tokenized, word2vec_model)

In [15]:
X_train_word2vec.shape

(159606, 300)

In [16]:
# Define the LightGBM model with your best settings
best_model = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    metric="multi_logloss",
    is_unbalance=True,
    class_weight="balanced",
    reg_alpha=0.01215570124339254,  # L1 regularization
    reg_lambda=0.07146814041554751,  # L2 regularization,
    learning_rate=0.03721005633871337,
    n_estimators=997,
    max_depth=9,
    num_leaves=133,
    min_child_samples=31,
    colsample_bytree=0.9234118017408489,
    subsample=0.8740422770140972
)

In [17]:
# Train the LightGBM model on Word2Vec embeddings
best_model.fit(X_train_word2vec, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.228915 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 76500
[LightGBM] [Info] Number of data points in the train set: 159606, number of used features: 300
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


,num_leaves,133
,max_depth,9
,learning_rate,0.03721005633871337
,n_estimators,997
,objective,'multiclass'
,class_weight,'balanced'
,min_child_samples,31
,subsample,0.8740422770140972
,colsample_bytree,0.9234118017408489
,reg_alpha,0.01215570124339254
,reg_lambda,0.07146814041554751


In [18]:
# Make predictions on the test data
y_pred = best_model.predict(X_test_word2vec)

In [19]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.76      0.77      0.76     13554
           1       0.76      0.74      0.75     17597
           2       0.57      0.59      0.58      8751

    accuracy                           0.72     39902
   macro avg       0.70      0.70      0.70     39902
weighted avg       0.72      0.72      0.72     39902



In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report
import numpy as np
import optuna
from sklearn.model_selection import cross_val_score

# Encode the target labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# Objective function for Optuna hyperparameter tuning
def objective(trial):
    # Suggest hyperparameters to be tuned
    params = {
        "objective": "multiclass",
        "num_class": 3,
        "metric": "multi_logloss",
        "is_unbalance": True,
        "class_weight": "balanced",
        "reg_alpha": 0.1,  # L1 regularization
        "reg_lambda": 0.1,  # L2 regularization
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "max_depth": trial.suggest_int("max_depth", 3, 20)
    }

    # Initialize the LightGBM model with suggested parameters
    model = LGBMClassifier(**params)

    # Perform cross-validation to evaluate the model performance
    scores = cross_val_score(model, X_train_word2vec, y_train_encoded, cv=3, scoring='accuracy')

    # Return the mean accuracy score across folds
    return scores.mean()

# Create an Optuna study to optimize the objective function
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)  # Run 30 trials

# Print the best trial
print("Best trial:")
best_params = study.best_trial.params
print(best_params)

# Train the LightGBM model with the best parameters
best_model = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    metric="multi_logloss",
    is_unbalance=True,
    class_weight="balanced",
    reg_alpha=0.1,
    reg_lambda=0.1,
    **best_params
)

best_model.fit(X_train_word2vec, y_train_encoded)

# Make predictions on the test data
y_pred = best_model.predict(X_test_word2vec)

[I 2026-08-20 14:27:01,413] A new study created in memory with name: no-name-35207a81-aab0-4112-8b66-38fec48767a8


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.147656 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 76500
[LightGBM] [Info] Number of data points in the train set: 106404, number of used features: 300
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.102506 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 76500
[LightGBM] [Info] Number of data points in the train set: 106404, number of used features: 300
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.1050

[I 2026-08-20 14:36:29,521] Trial 0 finished with value: 0.6394747064646692 and parameters: {'n_estimators': 836, 'learning_rate': 0.011896598613587708, 'max_depth': 17}. Best is trial 0 with value: 0.6394747064646692.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.085311 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 76500
[LightGBM] [Info] Number of data points in the train set: 106404, number of used features: 300
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.072942 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 76500
[LightGBM] [Info] Number of data points in the train set: 106404, number of used features: 300
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.1033

[I 2026-08-20 14:40:15,883] Trial 1 finished with value: 0.6633585203563775 and parameters: {'n_estimators': 339, 'learning_rate': 0.0549525674024579, 'max_depth': 15}. Best is trial 1 with value: 0.6633585203563775.


...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.084268 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 76500
[LightGBM] [Info] Number of data points in the train set: 106404, number of used features: 300
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.078223 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 76500
[LightGBM] [Info] Number of data points in the train set: 106404, number of used features: 300
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.0834

[I 2026-08-20 15:08:15,320] Trial 8 finished with value: 0.6375950778792777 and parameters: {'n_estimators': 207, 'learning_rate': 0.045241823126871174, 'max_depth': 19}. Best is trial 6 with value: 0.6971605077503352.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.100867 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 76500
[LightGBM] [Info] Number of data points in the train set: 106404, number of used features: 300
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] 

[I 2026-08-20 15:11:39,605] Trial 9 finished with value: 0.6791098079019586 and parameters: {'n_estimators': 735, 'learning_rate': 0.08874067427734357, 'max_depth': 4}. Best is trial 6 with value: 0.6971605077503352.


Best trial:
{'n_estimators': 895, 'learning_rate': 0.24693227009675653, 'max_depth': 18}
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.115300 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 76500
[LightGBM] [Info] Number of data points in the train set: 159606, number of used features: 300
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612


In [22]:
# Print classification report
print(classification_report(y_test_encoded, y_pred))

              precision    recall  f1-score   support

           0       0.75      0.76      0.75     13554
           1       0.75      0.74      0.75     17597
           2       0.56      0.56      0.56      8751

    accuracy                           0.71     39902
   macro avg       0.69      0.69      0.69     39902
weighted avg       0.71      0.71      0.71     39902

